# Nested Cross-Validation

Notebook for the course [Master Hyperparameter Optimization for Tabular Learning](http://www.trainindata.com/p/master-hyperparameter-optimization-for-tabular-learning)

In this notebook, we will implement nested cross-validation to both select the best hyperparameters and obtain a better estimate of the generalization error of the final model.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import (
    KFold,
    GridSearchCV,
    train_test_split,
)

In [2]:
# if you want more information about the dataset for this demo:

# scikit-learn dataset
# https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-dataset

# dataset information: UCI Machine Learning Repository
# https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic)
    
# in short, classification problem, trying to predict whether the tumor
# is malignant or benign

# load dataset
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
y = y.map({0:1, 1:0})

X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
# percentage of benign (0) and malign tumors (1)

y.value_counts() / len(y)

target
0    0.627417
1    0.372583
Name: count, dtype: float64

In [4]:
# split dataset into a train and test set

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

X_train.shape, X_test.shape

((398, 30), (171, 30))

## Nested Cross-Validation

In [5]:
def nested_cross_val(model, grid):

    # configure the outer loop cross-validation procedure
    cv_outer = KFold(n_splits=5, shuffle=True, random_state=1)

    # configure the inner loop cross-validation procedure
    cv_inner = KFold(n_splits=5, shuffle=True, random_state=1)

    # enumerate splits
    outer_scores = list()
    inner_mean_scores = list()
    inner_std_scores = list()

    for train_ix, test_ix in cv_outer.split(X_train):

        # split data
        xtrain, xtest = X_train.iloc[train_ix], X_train.iloc[test_ix]
        ytrain, ytest = y_train.iloc[train_ix], y_train.iloc[test_ix]

        # define search
        search = GridSearchCV(
            model, grid, scoring='accuracy', cv=cv_inner, refit=True)

        # execute search
        search.fit(xtrain, ytrain)

        # evaluate model on the hold out dataset
        yhat = search.predict(xtest)

        # evaluate the model
        outer_score = accuracy_score(ytest, yhat)
        inner_mean = search.best_score_
        inner_std = search.cv_results_['std_test_score'][search.best_index_]

        # store the result
        outer_scores.append(outer_score)
        inner_mean_scores.append(inner_mean)
        inner_std_scores.append(inner_std)
        outer_std = np.std(outer_scores)


    # summarize the estimated performance of the model
    print()
    print('outer accuracy: %.3f +- %.3f' %
          (np.mean(outer_scores), np.std(outer_scores)))
    print('inner accuracy: %.3f; mean inner-fold std: %.3f' %
          (np.mean(inner_mean_scores), np.mean(inner_std_scores)))

    return search.fit(X_train, y_train)

## Logistic Regression

In [6]:
# Logistic Regression
logit = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        l1_ratio=0, C=1, solver='saga', random_state=4, max_iter=10000
    )),
])

# hyperparameter space
logit_param = dict(
    classifier__l1_ratio=[0, 0.5, 1],
    classifier__C=[0.1, 1, 10],
)

In [7]:
logit_search = nested_cross_val(logit, logit_param)


outer accuracy: 0.980 +- 0.013
inner accuracy: 0.980; mean inner-fold std: 0.012


The inner-loop standard deviation describes variation among validation folds for the selected configuration within each outer split. The outer-loop standard deviation describes variation in generalization performance across held-out outer folds and therefore reflects the variability of the full model-selection procedure.

In [8]:
# let's get the predictions

X_train_preds = logit_search.predict(X_train)
X_test_preds = logit_search.predict(X_test)

# let's examine the accuracy
print('Train accuracy: ', accuracy_score(y_train, X_train_preds))
print('Test accuracy: ', accuracy_score(y_test, X_test_preds))

Train accuracy:  0.9899497487437185
Test accuracy:  0.9766081871345029


Compare the held-out test accuracy with the mean and standard deviation estimated by the outer loop. The outer results evaluate the complete model-selection procedure on data that was not used by the corresponding inner search.

## Random Forests

In [9]:
rf_param = dict(
    n_estimators=[10, 50, 100, 200],
    min_samples_split=[0.1, 0.3, 0.5, 1.0],
    max_depth=[1,2,3,None],
    )

rf = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=2,
    max_depth=3,
    random_state=0,
    n_jobs=-1,
    )

In [10]:
rf_search = nested_cross_val(rf, rf_param)


outer accuracy: 0.950 +- 0.025
inner accuracy: 0.950; mean inner-fold std: 0.020


The inner-loop standard deviation describes variation among validation folds for the selected configuration within each outer split. The outer-loop standard deviation describes variation in generalization performance across held-out outer folds and therefore reflects the variability of the full model-selection procedure.

In [11]:
# let's get the predictions

X_train_preds = rf_search.predict(X_train)
X_test_preds = rf_search.predict(X_test)

# let's examine the accuracy
print('Train accuracy: ', accuracy_score(y_train, X_train_preds))
print('Test accuracy: ', accuracy_score(y_test, X_test_preds))

Train accuracy:  0.9824120603015075
Test accuracy:  0.9473684210526315
